# Полный цикл гибридной системы

Этот ноутбук демонстрирует полный цикл работы гибридной системы DeepFM+SVD++ + Dueling DQN:

1. **Подготовка данных** - загрузка датасета (ITM-Rec или OULAD)
2. **Обучение DeepFM+SVD++** - многокритериальная модель
3. **Обучение DQN** - агент обучения с подкреплением
4. **Тестирование гипотез H1/H2/H3** - проверка эффективности
5. **Визуализация траекторий** - анализ поведения агента

## На выходе

- Полностью обученные модели
- Метрики сравнения с базовыми методами
- Долгосрочные метрики оценки
- Адаптивность к изменениям контекста
- Ablation-анализ компонент
- Графики и таблицы результатов

Среднее время выполнения: ~10-15 минут (на CPU) или ~2-3 минуты (на GPU).

---

## Содержание

1. Импорт и конфигурация
2. Загрузка данных
3. Обучение DeepFM+SVD++
4. Обучение DQN
5. Тестирование гипотез (H1/H2/H3)
6. Визуализация траекторий
7. Итоговый отчет


## 1 Импорт и конфигурация

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import api
from src.utils.helpers import set_seed, get_device

# Настройка визуализации
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 5)

# Используем неинтерактивный бэкенд для Jupyter
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*FigureCanvasAgg.*')

# Воспроизводимость
set_seed(42)

print("Импорты готовы")
print(f"Device: {get_device()}")

Импорты готовы
Device: cpu


### Конфигурация

Выберите датасет и настройки ниже:

In [2]:
# ========== ВЫБОР ДАТАСЕТА И ПАРАМЕТРОВ ==========
DATASET = "oulad"  # "itmrec" или "oulad"
EPOCHS_STATIC = 10  # количество эпох для DeepFM (рекомендуется 50 для продакшена)
EPISODES_DQN = 100  # количество эпизодов для DQN (рекомендуется 500+ для продакшена)
N_EVAL_USERS = 20   # количество пользователей для тестирования

print(f"Конфигурация:")
print(f"  Dataset: {DATASET}")
print(f"  DeepFM epochs: {EPOCHS_STATIC}")
print(f"  DQN episodes: {EPISODES_DQN}")
print(f"  Eval users: {N_EVAL_USERS}")

# ========== ПОДГОТОВКА РАБОЧЕЙ ДИРЕКТОРИИ ==========
config = api.build_config(DATASET, yaml_path=str(ROOT / "configs" / f"{DATASET}.yaml"))
# Переопределяем параметры обучения
config['model']['deepfm']['n_epochs'] = EPOCHS_STATIC
config['training']['dqn']['n_episodes'] = EPISODES_DQN

run_dir = api.prepare_run(config, run_name=f"{DATASET}_quickstart")
print(f"\nRun directory: {run_dir}")

Конфигурация:
  Dataset: oulad
  DeepFM epochs: 10
  DQN episodes: 100
  Eval users: 20
2026-05-17 22:41:57 | INFO | rec_sys_edu | Запуск: results\oulad_quickstart_20260517_224157

Run directory: results\oulad_quickstart_20260517_224157


## 2 Загрузка данных

In [3]:
print(f"Загрузка датасета {DATASET}...")
bundle = api.load_dataset_bundle(DATASET, config=config)

print(f"\nДатасет загружен:")
print(f"  Пользователей: {bundle.n_users}")
print(f"  Объектов: {bundle.n_items}")
print(f"  Рейтингов: {len(bundle.ratings)}")
print(f"  State dim: {bundle.state_dim}")

# Показываем несколько записей
print(f"\nПервые записи рейтингов:")
display(bundle.ratings.head())

Загрузка датасета oulad...
2026-05-17 22:45:51 | INFO | rec_sys_edu | OULAD bundle: users=29632, items=18, step_rows=2318825

Датасет загружен:
  Пользователей: 29632
  Объектов: 18
  Рейтингов: 2318825
  State dim: 96

Первые записи рейтингов:


,UserID_encoded,ItemID_encoded,week_index,Mastery,Engagement,SelfRegulation,Outcome,Module_encoded,Presentation_encoded
0,0,0,-2,0.46,0.106746,0.91219,0.562,5,1
1,0,3,-2,0.46,0.235919,0.91219,0.562,5,1
2,0,1,-2,0.46,0.253234,0.91219,0.562,5,1
3,0,5,-2,0.46,0.062943,0.91219,0.562,5,1
4,0,2,-2,0.46,0.317122,0.91219,0.562,5,1


## 3️ Обучение DeepFM+SVD++

In [4]:
print(f"Обучение DeepFM+SVD++ ({EPOCHS_STATIC} эпох)...\n")

static_result = api.train_static(
    DATASET,
    config=config,
    bundle=bundle,
    run_dir=run_dir
)

deepfm_model = static_result["model"]
deepfm_history = static_result["history"]
deepfm_ckpt = Path(deepfm_history["best_checkpoint"])

print(f"  DeepFM обучена")
print(f"  Best val loss: {deepfm_history.get('best_val_loss', 'N/A'):.4f}")
print(f"  Checkpoint: {deepfm_ckpt.name}")

# Визуализация истории обучения
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
if deepfm_history.get('train_losses'):
    ax.plot(deepfm_history['train_losses'], label='Train Loss', marker='o', alpha=0.7)
if deepfm_history.get('val_losses'):
    ax.plot(deepfm_history['val_losses'], label='Val Loss', marker='s', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('DeepFM+SVD++ Training History')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(run_dir / "figures" / "deepfm_training.png", dpi=100, bbox_inches='tight')
plt.show()

Обучение DeepFM+SVD++ (10 эпох)...

2026-05-17 22:45:55 | INFO | rec_sys_edu | Static model: 29632 users, 18 items, 4 heads
2026-05-17 22:47:12 | INFO | rec_sys_edu | Epoch 1/10 - train_loss=0.0077, val_loss=0.0033
2026-05-17 22:47:12 | INFO | rec_sys_edu | Saved best model: results\oulad_quickstart_20260517_224157\models\deepfm_oulad_best.pth (val_loss=0.0033)
2026-05-17 22:48:42 | INFO | rec_sys_edu | Epoch 2/10 - train_loss=0.0037, val_loss=0.0029
2026-05-17 22:48:42 | INFO | rec_sys_edu | Saved best model: results\oulad_quickstart_20260517_224157\models\deepfm_oulad_best.pth (val_loss=0.0029)
2026-05-17 22:50:12 | INFO | rec_sys_edu | Epoch 3/10 - train_loss=0.0033, val_loss=0.0028
2026-05-17 22:50:12 | INFO | rec_sys_edu | Saved best model: results\oulad_quickstart_20260517_224157\models\deepfm_oulad_best.pth (val_loss=0.0028)
2026-05-17 22:51:43 | INFO | rec_sys_edu | Epoch 4/10 - train_loss=0.0032, val_loss=0.0026
2026-05-17 22:51:43 | INFO | rec_sys_edu | Saved best model: resu

## 4️ Обучение DQN агента

In [5]:
print(f"Обучение Dueling DQN ({EPISODES_DQN} эпизодов)...\n")

dqn_result = api.train_dqn(
    DATASET,
    config=config,
    bundle=bundle,
    deepfm_model=deepfm_model,
    run_dir=run_dir
)

dqn_trainer = dqn_result["trainer"]
dqn_history = dqn_result["history"]
dqn_ckpt = Path(dqn_result["checkpoint"])

print(f"  DQN обучен")
print(f"  Checkpoint: {dqn_ckpt.name}")

# Визуализация обучения
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Награды
if dqn_history.get('training_rewards'):
    axes[0].plot(dqn_history['training_rewards'], alpha=0.7, linewidth=0.8)
    axes[0].set_title('Episode Rewards')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Total Reward')
    axes[0].grid(True, alpha=0.3)

# Потери
if dqn_history.get('training_losses'):
    axes[1].plot(dqn_history['training_losses'], alpha=0.7, color='orange', linewidth=0.8)
    axes[1].set_title('Training Loss')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)

# Epsilon decay
if dqn_history.get('epsilon_values'):
    axes[2].plot(dqn_history['epsilon_values'], alpha=0.7, color='green', linewidth=0.8)
    axes[2].set_title('Epsilon (Exploration Rate)')
    axes[2].set_xlabel('Episode')
    axes[2].set_ylabel('Epsilon')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / "figures" / "dqn_training.png", dpi=100, bbox_inches='tight')
plt.show()

Обучение Dueling DQN (100 эпизодов)...

Инициализация тренера на cpu
  Gamma: 0.95
  Learning rate: 0.0005
  Batch size: 128
  Epsilon: 1.0 -> 0.05
Запуск обучения на 100 эпизодов...


 50%|█████     | 50/100 [00:14<00:24,  2.01it/s]


Эпизод 50:
  Награда обучения: 1.043
  Средняя оценка: 2.808
  Epsilon: 0.617


100%|██████████| 100/100 [00:30<00:00,  3.29it/s]


Эпизод 100:
  Награда обучения: 0.996
  Средняя оценка: 1.612
  Epsilon: 0.358

Обучение завершено!


Чекпоинт сохранен: results\oulad_quickstart_20260517_224157\models\dqn_oulad_checkpoint.pth
2026-05-17 23:07:27 | INFO | rec_sys_edu | DQN training completed: episodes=100, final epsilon=0.358, mean_reward=1.365
  DQN обучен
  Checkpoint: dqn_oulad_checkpoint.pth


## 5️ Тестирование гипотез H1/H2/H3

In [6]:
print(f"Запуск тестов гипотез H1/H2/H3...\n")

eval_result = api.evaluate_system(
    DATASET,
    hypothesis="all",
    config=config,
    bundle=bundle,
    deepfm_model=deepfm_model,
    dqn_agent=dqn_trainer.agent,
    run_dir=run_dir
)

results = eval_result["results"]
print(f"Гипотезы протестированы")
print(f"  Результаты сохранены в: {run_dir}/tables/")
print(f"\nКлючевые метрики:")
print(f"Гипотезы в results: {list(results.keys())}")

Запуск тестов гипотез H1/H2/H3...

2026-04-23 16:33:04 | INFO | rec_sys_edu | Гипотеза H1: долгосрочная полезность (CDR / Retention / Slope / Coverage)
2026-04-23 16:33:04 | INFO | rec_sys_edu | LongTerm: запускаем DQN
2026-04-23 16:33:13 | INFO | rec_sys_edu | LongTerm: запускаем Random
2026-04-23 16:33:14 | INFO | rec_sys_edu | LongTerm: запускаем Popularity
2026-04-23 16:33:23 | INFO | rec_sys_edu | Гипотеза H2: адаптивность и стабильность по стратам
2026-04-23 16:33:36 | INFO | rec_sys_edu | Гипотеза H3: баланс новизны и релевантности
2026-04-23 16:33:38 | INFO | rec_sys_edu | Ablation variant 'full': mean_reward=1.076 novelty=0.348 diversity=0.511
2026-04-23 16:33:41 | INFO | rec_sys_edu | Ablation variant 'no_novelty': mean_reward=0.513 novelty=0.263 diversity=0.445
2026-04-23 16:33:43 | INFO | rec_sys_edu | Ablation variant 'novelty_popularity': mean_reward=1.163 novelty=0.275 diversity=0.453
Гипотезы протестированы
  Результаты сохранены в: results\oulad_quickstart_20260423_143

### H1: Долгосрочная полезность (CDR, Retention Rate, Learning Slope, Final Coverage)

In [7]:
h1_data = results.get("H1", {})
if h1_data:
    print("\nH1 - Долгосрочная полезность (DQN vs Random / Popularity / DeepFM-SVD++):")
    per_model = h1_data.get("per_model", {})
    rows = []
    for model_name, block in per_model.items():
        mean_vals = block.get("mean", {})
        row = {"model": model_name}
        for key in ("CDR", "Retention_Rate", "Learning_Slope", "Final_Coverage"):
            row[key] = mean_vals.get(key, 0.0)
        for key, val in mean_vals.items():
            if key.startswith("Precision@") or key.startswith("Recall@") or key.startswith("F1@"):
                row[key] = val
        rows.append(row)
    if rows:
        df_h1 = pd.DataFrame(rows).set_index("model")
        print(df_h1.round(4))
    sig = h1_data.get("significance_cdr", {})
    if sig:
        print("\nСтатистика CDR vs DQN (Welch t-test):")
        print(pd.DataFrame(sig).T.round(4))
    print(f"\nПорог Retention t_r = {h1_data.get('retention_threshold', 0.5)}")

    # Визуализация H1: CDR/Retention/Slope/Coverage/Precision по моделям
    if rows:
        fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
        metrics_plot = [
            ("CDR", "Кумулятивная дисконтированная награда (CDR)", "tab:blue"),
            ("Retention_Rate", "Retention Rate", "tab:green"),
            ("Learning_Slope", "Learning Slope", "tab:purple"),
            ("Final_Coverage", "Final Coverage", "tab:orange"),
        ]
        models = df_h1.index.tolist()
        for ax, (col, title, color) in zip(axes, metrics_plot):
            values = df_h1[col].to_numpy() if col in df_h1.columns else np.zeros(len(models))
            ax.bar(models, values, color=color, alpha=0.85)
            ax.set_title(title)
            ax.tick_params(axis="x", rotation=30)
            if col == "Learning_Slope":
                ax.axhline(y=0, color="red", linestyle="--", alpha=0.4)
            ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.savefig(run_dir / "figures" / "h1_long_term_metrics.png", dpi=100, bbox_inches='tight')
        plt.show()
else:
    print("H1 данные не доступны")


H1 - Долгосрочная полезность vs baselines:
               CDR  Retention_Rate  Learning_Slope  Final_Coverage  \
model                                                                
DQN         0.8850          0.6283         -0.0080          0.6300   
Random     -0.1546          0.3705         -0.0038          0.1556   
Popularity  2.7366          0.7800         -0.0010          0.0556   

            Precision@10  Recall@10   F1@10  
model                                        
DQN               0.6080     0.5008  0.5485  
Random            0.3586     0.5371  0.4273  
Popularity        0.0780     0.7800  0.1418  

Статистика CDR vs DQN:
               t_stat p_value significant mean_diff
Random      11.849857     0.0        True  1.039614
Popularity  -7.666371     0.0        True -1.851622

Порог Retention τ_r = 0.0


### H2: Адаптивность и стабильность (AdaptabilityScore, σ_P, σ_R по стратам)

In [8]:
h2_data = results.get("H2", {})
if h2_data:
    print("\nH2 - Адаптивность и стабильность:")
    scalar_keys = [
        "adaptability_score",
        "learning_slope",
        "precision_stability",
        "recall_stability",
        "per_user_precision",
        "per_user_recall",
    ]
    scalar_row = {k: h2_data.get(k, 0.0) for k in scalar_keys}
    print(pd.DataFrame([scalar_row]).round(4).T.rename(columns={0: "value"}))

    by_strata = h2_data.get("by_strata", {})
    if by_strata:
        print("\nСтратификация по контексту/демографии (сегментный анализ):")
        for stratum_col, stratum_table in by_strata.items():
            df_strat = pd.DataFrame(stratum_table).T
            print(f"\n  {stratum_col}:")
            print(df_strat.round(4))

    # Ablation-анализ по конфигурациям состояния (full_state/no_context/no_demo/...)
    state_abl = h2_data.get("state_ablation", {})
    if state_abl:
        print("\nH2 - Ablation по конфигурациям состояния:")
        rows = []
        for mode, payload in state_abl.items():
            rows.append({
                "state_ablation": mode,
                "AdaptabilityScore": payload.get("adaptability_score", 0.0),
                "PrecisionStability": payload.get("precision_stability", 0.0),
                "RecallStability": payload.get("recall_stability", 0.0),
                "Precision@K": payload.get("per_user_precision", 0.0),
                "Recall@K": payload.get("per_user_recall", 0.0),
                "CDR": payload.get("cdr", 0.0),
            })
        df_state_abl = pd.DataFrame(rows).set_index("state_ablation")
        print(df_state_abl.round(4))

    # График CDR между моделями H1 (сводка)
    h1_data_local = results.get("H1", {}).get("per_model", {})
    if h1_data_local:
        cdr_vals = {m: block.get("mean", {}).get("CDR", 0.0) for m, block in h1_data_local.items()}
        fig, ax = plt.subplots(figsize=(10, 5))
        pd.Series(cdr_vals).plot(kind='bar', ax=ax, color=['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple'][:len(cdr_vals)])
        ax.set_title('Кумулятивная дисконтированная награда по моделям (H1)')
        ax.set_ylabel('CDR')
        ax.set_xlabel('Метод')
        ax.grid(True, alpha=0.3, axis='y')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig(run_dir / "figures" / "h1_cdr_comparison.png", dpi=100, bbox_inches='tight')
        plt.show()
else:
    print("H2 данные не доступны")


H2 - Адаптивность и стабильность:
                      value
adaptability_score   0.8808
learning_slope      -0.0096
precision_stability  0.0765
recall_stability     0.1619
per_user_precision   0.2180
per_user_recall      0.4553

Стратификация по контексту/демографии:

  Module_encoded:
   precision  recall  mean_reward  n_users
0     0.1000  0.3750       0.0082      4.0
1     0.2429  0.6508       0.0242      7.0
2     0.1750  0.3682      -0.0285      8.0
3     0.2727  0.3903      -0.0565     11.0
4     0.2429  0.4388       0.0211      7.0
5     0.2455  0.5817       0.0044     11.0
6     0.0000  0.0000       0.0316      2.0

  Presentation_encoded:
   precision  recall  mean_reward  n_users
0     0.1833  0.2870      -0.1017      6.0
1     0.2154  0.4731       0.0244     13.0
2     0.2500  0.4559      -0.0206     12.0
3     0.2105  0.4959       0.0080     19.0


### H3: Ablation анализ

In [9]:
h3_data = results.get("H3", {})
if h3_data:
    print("\nH3 - Ablation новизны (full / no_novelty / novelty_popularity):")
    ablation_list = h3_data.get("ablation", [])
    extra_metrics = h3_data.get("extra_metrics", {})
    if ablation_list:
        rows = {}
        for item in ablation_list:
            variant = item.get('variant', 'unknown')
            extra = extra_metrics.get(variant, {})
            rows[variant] = {
                'Mean_Reward': item.get('mean_reward', 0),
                'Std_Reward': item.get('std_reward', 0),
                'Novelty': item.get('mean_novelty', 0),
                'Diversity': item.get('mean_diversity', 0),
                'Coverage': extra.get('coverage', 0.0),
                'Precision@10': extra.get('precision@10', 0.0),
                'Recall@10': extra.get('recall@10', 0.0),
                'F1@10': extra.get('f1@10', 0.0),
                'CDR': extra.get('cdr', 0.0),
            }
        df_ablation = pd.DataFrame(rows).T
        print("\n", df_ablation.round(4))

        pairwise = h3_data.get('pairwise_vs_full', {})
        if pairwise:
            print("\nПарный тест (Welch t-test) vs full:")
            print(pd.DataFrame(pairwise).T.round(4))

        # Визуализация всех ключевых метрик H3
        variants = list(df_ablation.index)
        metrics_plot = [
            ('Mean_Reward', 'Средняя награда эпизода', 'tab:blue'),
            ('Novelty', 'Novelty', 'tab:orange'),
            ('Diversity', 'Diversity', 'tab:green'),
            ('Coverage', 'Coverage', 'tab:red'),
            ('Precision@10', 'Precision@10', 'tab:purple'),
            ('Recall@10', 'Recall@10', 'tab:brown'),
            ('F1@10', 'F1@10', 'tab:pink'),
            ('CDR', 'Кумулятивная дисконтированная награда (CDR)', 'tab:gray'),
        ]
        fig, axes = plt.subplots(2, 4, figsize=(20, 8))
        for ax, (col, title, color) in zip(axes.flatten(), metrics_plot):
            values = df_ablation[col].to_numpy()
            ax.bar(variants, values, color=color, alpha=0.85)
            ax.set_title(title)
            ax.tick_params(axis='x', rotation=30)
            ax.grid(True, alpha=0.3, axis='y')
        fig.suptitle('H3: сравнение конфигураций награды (ablation новизны)', fontsize=13)
        plt.tight_layout()
        plt.savefig(run_dir / "figures" / "h3_ablation.png", dpi=100, bbox_inches='tight')
        plt.show()
    else:
        print("Ablation данные пусты")
else:
    print("H3 данные не доступны")


H3 - Ablation новизны и состояния:

                     Mean_Reward  Std_Reward  Mean_Novelty  Mean_Diversity
full                     1.0759      0.6327        0.3484          0.5112
no_novelty               0.5132      0.3756        0.2628          0.4447
novelty_popularity       1.1626      0.5337        0.2754          0.4525


## 6️ Визуализация траекторий

In [10]:
print("Визуализирую траектории агентов...\n")

env = api.build_environment(bundle, deepfm_model, config)

# Собираем траектории
sample_users = np.random.choice(bundle.ratings["UserID_encoded"].unique(), min(8, bundle.n_users), replace=False)

def collect_trajectory(env, agent, user_id, max_steps=30):
    state = env.reset(user_id=int(user_id))
    actions, rewards = [], []
    for _ in range(max_steps):
        mask = env.get_action_mask() if hasattr(env, "get_action_mask") else None
        action = int(agent.get_action(state, epsilon=0.01, action_mask=mask))
        next_state, reward, done, _ = env.step(action)
        actions.append(action)
        rewards.append(float(reward))
        state = next_state
        if done:
            break
    return {"user_id": int(user_id), "actions": actions, "rewards": rewards}

trajectories_dqn = [collect_trajectory(env, dqn_trainer.agent, u) for u in sample_users]
print(f"Собрано {len(trajectories_dqn)} траекторий")

# Единые подписи и цвета пользователей для всех подграфиков (чтобы можно было сопоставлять)
user_labels = [f"user_{traj['user_id']}" for traj in trajectories_dqn]
user_ids = [traj['user_id'] for traj in trajectories_dqn]
cmap = plt.cm.get_cmap('tab10', max(len(trajectories_dqn), 1))
user_colors = [cmap(i) for i in range(len(trajectories_dqn))]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# [0][0] Кумулятивная награда
ax = axes[0, 0]
for traj, label, color in zip(trajectories_dqn, user_labels, user_colors):
    cumsum = np.cumsum(traj['rewards'])
    ax.plot(cumsum, alpha=0.8, label=label, color=color)
ax.set_title('Кумулятивная награда по шагам')
ax.set_xlabel('Шаг')
ax.set_ylabel('Кумулятивная награда')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)

# [0][1] Распределение пошаговых наград
ax = axes[0, 1]
rewards_per_step = [r for traj in trajectories_dqn for r in traj['rewards']]
ax.hist(rewards_per_step, bins=30, alpha=0.7, color='skyblue')
ax.set_title('Распределение пошаговых наград')
ax.set_xlabel('Награда за шаг')
ax.set_ylabel('Частота')
ax.axvline(np.mean(rewards_per_step), color='red', linestyle='--', label=f'Среднее: {np.mean(rewards_per_step):.3f}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# [1][0] Длина эпизода для каждого пользователя
ax = axes[1, 0]
episode_lengths = [len(traj['rewards']) for traj in trajectories_dqn]
xs = np.arange(len(episode_lengths))
ax.bar(xs, episode_lengths, color=user_colors)
ax.set_xticks(xs)
ax.set_xticklabels(user_labels, rotation=30, ha='right', fontsize=8)
ax.set_title('Длина эпизода по пользователям')
ax.set_xlabel('Пользователь')
ax.set_ylabel('Число шагов')
ax.grid(True, alpha=0.3, axis='y')

# [1][1] Суммарная награда по пользователям
ax = axes[1, 1]
total_rewards = [sum(traj['rewards']) for traj in trajectories_dqn]
ax.bar(xs, total_rewards, color=user_colors)
ax.set_xticks(xs)
ax.set_xticklabels(user_labels, rotation=30, ha='right', fontsize=8)
ax.set_title('Суммарная награда за эпизод')
ax.set_xlabel('Пользователь')
ax.set_ylabel('Суммарная награда')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(run_dir / "figures" / "trajectories_analysis.png", dpi=100, bbox_inches='tight')
plt.show()

Визуализирую траектории агентов...

Собрано 8 траекторий


## Итоговый отчет

In [11]:
print("\n" + "="*70)
print("ПОЛНЫЙ ЦИКЛ ЗАВЕРШЕН")
print("="*70)

print(f"\nВсе результаты сохранены в: {run_dir}\n")

# Список файлов
tables_dir = run_dir / "tables"
figures_dir = run_dir / "figures"

if tables_dir.exists():
    print("Таблицы результатов:")
    for f in sorted(tables_dir.glob("*.json")):
        print(f"  • {f.name}")

if figures_dir.exists():
    print("\nГрафики:")
    for f in sorted(figures_dir.glob("*.png")):
        print(f"  • {f.name}")

print(f"\nConfig: {run_dir / 'config.yaml'}")
print(f"Logs: {run_dir / 'logs'}")


ПОЛНЫЙ ЦИКЛ ЗАВЕРШЕН

Все результаты сохранены в: results\oulad_quickstart_20260423_143412

Таблицы результатов:
  • deepfm_history.json
  • dqn_history.json
  • evaluation_summary.json
  • h1_long_term.json
  • h2_adaptability.json
  • h3_novelty_ablation.json

Графики:
  • deepfm_training.png
  • dqn_training.png
  • h1_cdr_comparison.png
  • h2_coverage_novelty.png
  • h2_trajectories.png
  • h3_ablation.png
  • trajectories_analysis.png

Config: results\oulad_quickstart_20260423_143412\config.yaml
Logs: results\oulad_quickstart_20260423_143412\logs
